## 4. Choose Final Model and Hyperparameter Tuning

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import average_precision_score, roc_auc_score

from src.utils import load_historical, make_pipeline

In [2]:
X, y, df = load_historical("../data/historical_data.csv")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(9600, 16) (2400, 16)
0.10197916666666666 0.10208333333333333


In [3]:
final_model = LogisticRegression(max_iter=2000, solver="liblinear")

pipe = make_pipeline(final_model, X_train)

param_grid = {
    "model__C": [0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1
)

In [4]:
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV AP:", grid.best_score_)

Best params: {'model__C': 10, 'model__class_weight': None}
Best CV AP: 0.2842212848223595


In [5]:
best_pipe = grid.best_estimator_

y_proba_test = best_pipe.predict_proba(X_test)[:, 1]

print("Test AP:", average_precision_score(y_test, y_proba_test))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba_test))

Test AP: 0.27602488309617335
Test ROC-AUC: 0.7399327619678961


### Hyperparameter tuning

We selected Logistic Regression as the final model because it achieved the best Average Precision during model comparison.

To improve the ranking quality, we tuned two hyperparameters:

- **C**: controls the strength of regularization. A larger value allows the model to fit the data more flexibly.
- **class_weight**: allows handling class imbalance.

We optimized these parameters using GridSearchCV with Average Precision as the evaluation metric. This aligns with the Operations requirement of prioritizing high-risk cases while keeping the review queue manageable.


### Final model

Logistic Regression was selected as the final model because it achieved the highest Average Precision during model comparison.

We performed hyperparameter tuning using GridSearchCV and optimized the parameters **C** and **class_weight** to improve ranking performance.

The final model achieved a **Test Average Precision of ~0.276** and **ROC-AUC of ~0.74**, which provides a solid ranking of high-risk cases for prioritization.
